# Two POVs + Trainable Feature Extractor with Regression Head

In [1]:
import math
import datetime
from sklearn.metrics import mean_squared_error, r2_score
from itertools import product
from tqdm import tqdm
from src.constants import *
from src.helpers import *
from src.trainable_pipeline import *

seed_everything(RANDOM_STATE)

## Load Data

In [2]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
print("Samples shape:", samples.shape)
display(samples[["exp_id", "volume", "top_path", "side_path"]].head())

Samples shape: (140, 23)


,exp_id,volume,top_path,side_path
0,1,38,photos/top_view_images/P2090072.JPG,photos/side_view_images/P2090161.JPG
1,1,57,photos/top_view_images/P2090073.JPG,photos/side_view_images/P2090163.JPG
2,1,76,photos/top_view_images/P2090074.JPG,photos/side_view_images/P2090167.JPG
3,2,19,photos/top_view_images/P2090075.JPG,photos/side_view_images/P2090171.JPG
4,2,38,photos/top_view_images/P2090076.JPG,photos/side_view_images/P2090173.JPG


In [3]:
def evaluate_setting_nested_cv(samples_df, backbone_name, fusion_name, head_name, mode_name):
    y = samples_df["volume"].to_numpy(dtype=float)
    groups = samples_df["exp_id"].to_numpy()

    outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
    outer_splits = list(outer_cv.split(np.zeros((len(samples_df), 1)), y, groups))
    fold_records = []
    oof_pred = np.full(len(samples_df), np.nan, dtype=float)

    print("=" * 90)
    print(backbone_name, fusion_name, head_name, mode_name)
    print("=" * 90)

    for fold_idx, (train_idx, test_idx) in enumerate(
        tqdm(outer_splits,
            total=len(outer_splits),
            desc=f"{backbone_name} | {fusion_name} | {head_name} | {mode_name}",),
        start=1,
    ):
        train_groups = groups[train_idx]
        n_inner = min(INNER_SPLITS, len(np.unique(train_groups)))
        inner_cv = GroupKFold(n_splits=n_inner)

        best_cfg = None
        best_inner_mae = float("inf")

        for cfg in TRAINING_CONFIGS[mode_name]:
            inner_maes = []
            for inner_train_rel, inner_val_rel in inner_cv.split(np.zeros((len(train_idx), 1)), y[train_idx], train_groups):
                inner_train_idx = train_idx[inner_train_rel]
                inner_val_idx = train_idx[inner_val_rel]
                backbone, head, preprocess, inner_val_mae = fit_model(
                    samples_df=samples_df,
                    train_idx=inner_train_idx,
                    val_idx=inner_val_idx,
                    backbone_name=backbone_name,
                    fusion_name=fusion_name,
                    head_name=head_name,
                    mode_name=mode_name,
                    cfg=cfg,
                    head_cfg=HEAD_CONFIGS[head_name],
                    batch_size=BATCH_SIZE,
                    max_epochs=MAX_EPOCHS,
                    seed=RANDOM_STATE + fold_idx,
                    device=DEVICE,
                )
                inner_maes.append(inner_val_mae)
                del backbone, head
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            mean_inner_mae = float(np.mean(inner_maes))
            if mean_inner_mae < best_inner_mae:
                best_inner_mae = mean_inner_mae
                best_cfg = cfg

        final_train_idx, final_val_idx = make_group_train_val_split(train_idx, groups, seed=RANDOM_STATE + fold_idx)
        backbone, head, preprocess, _ = fit_model(
            samples_df=samples_df,
            train_idx=final_train_idx,
            val_idx=final_val_idx,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
            head_name=head_name,
            mode_name=mode_name,
            cfg=best_cfg,
            head_cfg=HEAD_CONFIGS[head_name],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            seed=RANDOM_STATE + fold_idx,
            device=DEVICE,
        )

        y_test_true, y_test_pred = predict_indices(samples_df, test_idx, preprocess, backbone_name, backbone, head, fusion_name, BATCH_SIZE, DEVICE)
        oof_pred[test_idx] = y_test_pred

        record = {
            "fold": fold_idx,
            "backbone": backbone_name,
            "fusion": fusion_name,
            "head": head_name,
            "mode": mode_name,
            "inner_MAE": best_inner_mae,
            "MAE": mean_absolute_error(y_test_true, y_test_pred),
            "MSE": mean_squared_error(y_test_true, y_test_pred),
            "RMSE": math.sqrt(mean_squared_error(y_test_true, y_test_pred)),
            "R2": r2_score(y_test_true, y_test_pred) if len(np.unique(y_test_true)) > 1 else np.nan,
            "best_cfg": best_cfg,
        }
        fold_records.append(record)

        print(f"Fold {fold_idx}: MAE={record['MAE']:.3f}, RMSE={record['RMSE']:.3f}, R2={record['R2']}, best={best_cfg}")

        del backbone, head
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return fold_records, oof_pred

def summarise_records(records):
    rows = []
    grouped = {}
    for rec in records:
        key = (rec["backbone"], rec["fusion"], rec["head"], rec["mode"])
        grouped.setdefault(key, []).append(rec)

    for (backbone, fusion, head, mode), folds in grouped.items():
        rows.append({
            "backbone": backbone,
            "fusion": fusion,
            "head": head,
            "mode": mode,
            "cv_mae_mean": np.mean([f["MAE"] for f in folds]),
            "cv_mae_std": np.std([f["MAE"] for f in folds]),
            "cv_rmse_mean": np.mean([f["RMSE"] for f in folds]),
            "cv_rmse_std": np.std([f["RMSE"] for f in folds]),
            "cv_r2_mean": np.mean([f["R2"] for f in folds]),
            "cv_r2_std": np.std([f["R2"] for f in folds]),
            "inner_mae_mean": np.mean([f["inner_MAE"] for f in folds]),
        })

    return pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)


In [4]:
all_fold_records = []
oof_store = {}

all_configs = list(product(
    BACKBONE_NAMES,
    FUSION_NAMES,
    HEAD_CONFIGS.keys(),
    TRAINING_CONFIGS.keys()
))


for backbone_name, fusion_name, head_name, mode_name in tqdm(all_configs, desc="Experiment configs"):
    print(f"\n=== {backbone_name} | {fusion_name} | {head_name} | {mode_name} ===")
    fold_records, oof_pred = evaluate_setting_nested_cv(
        samples_df=samples,
        backbone_name=backbone_name,
        fusion_name=fusion_name,
        head_name=head_name,
        mode_name=mode_name,
    )
    all_fold_records.extend(fold_records)
    oof_store[(backbone_name, fusion_name, head_name, mode_name)] = oof_pred

summary_df = summarise_records(all_fold_records)
summary_df

Experiment configs:   0%|          | 0/1 [00:00<?, ?it/s]


=== vit_b_16 | concat | mlp | last_stage ===
vit_b_16 concat mlp last_stage



vit_b_16 | concat | mlp | last_stage:   0%|          | 0/5 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:28<?, ?it/s, best=35.55, patience=6, val=35.55]

Epochs:   5%|▌         | 1/20 [00:28<08:52, 28.02s/it, best=35.55, patience=6, val=35.55]

Epochs:   5%|▌         | 1/20 [00:52<08:52, 28.02s/it, best=22.66, patience=6, val=22.66]

Epochs:  10%|█         | 2/20 [00:52<07:44, 25.78s/it, best=22.66, patience=6, val=22.66]

Epochs:  10%|█         | 2/20 [01:19<07:44, 25.78s/it, best=18.00, patience=6, val=18.00]

Epochs:  15%|█▌        | 3/20 [01:19<07:28, 26.36s/it, best=18.00, patience=6, val=18.00]

Epochs:  15%|█▌        | 3/20 [01:47<07:28, 26.36s/it, best=16.45, patience=6, val=16.45]

Epochs:  20%|██        | 4/20 [01:47<07:10, 26.93s/it, best=16.45, patience=6, val=16.45]

Epochs:  20%|██        | 4/20 [02:14<07:10, 26.93s/it, best=14.65, patience=6, val=14.65]

Epochs:  25%|██▌       | 5/20 [02:14<06:47, 27.14s/it, best=14.

Fold 1: MAE=8.284, RMSE=10.455, R2=0.7577469944953918, best={'head_lr': 0.001, 'backbone_lr': 1e-05, 'weight_decay': 0.0001}




Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:28<?, ?it/s, best=26.20, patience=6, val=26.20]

Epochs:   5%|▌         | 1/20 [00:28<09:03, 28.61s/it, best=26.20, patience=6, val=26.20]

Epochs:   5%|▌         | 1/20 [00:57<09:03, 28.61s/it, best=17.51, patience=6, val=17.51]

Epochs:  10%|█         | 2/20 [00:57<08:35, 28.66s/it, best=17.51, patience=6, val=17.51]

Epochs:  10%|█         | 2/20 [01:25<08:35, 28.66s/it, best=15.94, patience=6, val=15.94]

Epochs:  15%|█▌        | 3/20 [01:25<08:04, 28.51s/it, best=15.94, patience=6, val=15.94]

Epochs:  15%|█▌        | 3/20 [01:54<08:04, 28.51s/it, best=13.69, patience=6, val=13.69]

Epochs:  20%|██        | 4/20 [01:54<07:36, 28.55s/it, best=13.69, patience=6, val=13.69]

Epochs:  20%|██        | 4/20 [02:22<07:36, 28.55s/it, best=12.28, patience=6, val=12.28]

Epochs:  25%|██▌       | 5/20 [02:22<07:07, 28.50s/it, best=12.28, patience=6, val=12.28]

Epochs:  25%|██▌       | 5/20 [02:51<07:07, 28.5

...



Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:28<?, ?it/s, best=29.73, patience=6, val=29.73]

Epochs:   5%|▌         | 1/20 [00:28<09:03, 28.58s/it, best=29.73, patience=6, val=29.73]

Epochs:   5%|▌         | 1/20 [00:56<09:03, 28.58s/it, best=17.43, patience=6, val=17.43]

Epochs:  10%|█         | 2/20 [00:56<08:30, 28.35s/it, best=17.43, patience=6, val=17.43]

Epochs:  10%|█         | 2/20 [01:25<08:30, 28.35s/it, best=16.11, patience=6, val=16.11]

Epochs:  15%|█▌        | 3/20 [01:25<08:04, 28.50s/it, best=16.11, patience=6, val=16.11]

Epochs:  15%|█▌        | 3/20 [01:54<08:04, 28.50s/it, best=14.38, patience=6, val=14.38]

Epochs:  20%|██        | 4/20 [01:54<07:38, 28.65s/it, best=14.38, patience=6, val=14.38]

Epochs:  20%|██        | 4/20 [02:23<07:38, 28.65s/it, best=12.79, patience=6, val=12.79]

Epochs:  25%|██▌       | 5/20 [02:23<07:10, 28.69s/it, best=12.79, patience=6, val=12.79]

Epochs:  25%|██▌       | 5/20 [02:52<07:10, 28.6

Fold 4: MAE=7.028, RMSE=8.744, R2=0.8249538540840149, best={'head_lr': 0.001, 'backbone_lr': 1e-05, 'weight_decay': 0.0001}




Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:27<?, ?it/s, best=32.20, patience=6, val=32.20]

Epochs:   5%|▌         | 1/20 [00:27<08:49, 27.87s/it, best=32.20, patience=6, val=32.20]

Epochs:   5%|▌         | 1/20 [00:56<08:49, 27.87s/it, best=18.28, patience=6, val=18.28]

Epochs:  10%|█         | 2/20 [00:56<08:25, 28.10s/it, best=18.28, patience=6, val=18.28]

Epochs:  10%|█         | 2/20 [01:24<08:25, 28.10s/it, best=16.43, patience=6, val=16.43]

Epochs:  15%|█▌        | 3/20 [01:24<07:58, 28.17s/it, best=16.43, patience=6, val=16.43]

Epochs:  15%|█▌        | 3/20 [01:53<07:58, 28.17s/it, best=14.58, patience=6, val=14.58]

Epochs:  20%|██        | 4/20 [01:53<07:35, 28.47s/it, best=14.58, patience=6, val=14.58]

Epochs:  20%|██        | 4/20 [02:21<07:35, 28.47s/it, best=12.69, patience=6, val=12.69]

Epochs:  25%|██▌       | 5/20 [02:21<07:08, 28.55s/it, best=12.69, patience=6, val=12.69]

Epochs:  25%|██▌       | 5/20 [02:50<07:08, 28.5

Fold 5: MAE=6.736, RMSE=8.368, R2=0.8492558598518372, best={'head_lr': 0.001, 'backbone_lr': 1e-05, 'weight_decay': 0.0001}


,backbone,fusion,head,mode,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std,inner_mae_mean
0,vit_b_16,concat,mlp,last_stage,7.51261,0.780718,9.443722,1.283193,0.798448,0.055833,7.324183


In [5]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
name = f"results_trainable_performance_{timestamp}.csv"
summary_df.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)

Saved: results/results_trainable_performance_2026-03-27_09-37-19.csv


In [7]:
df = pd.DataFrame(all_fold_records)
name = f"results_trainable_performance_folds_{timestamp}.csv"
df.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)

Saved: results/results_trainable_performance_folds_2026-03-27_09-40-32.csv
